# ksw: demo2 full drill profiling

What does this notebook do 
Goes through the entire drill learnt so far
Precursors
- i)  load model and tokenizer
- ii)  create a batch of inputs
- iii) define functions for profiling inference and displaying tables
- iv) definire functions for model metrics calculation
THE REAL WORK
- i) profile original BASELINE MODEL for INFERENCE (not TRAINING, but only the inference)
- ii) apply pruning 
      - PRUNING: unstructured 30% pruning on the smallest weights. uses pytorch native utils- torch.nn.utils.prune \
- iii) apply various types of pruning
      - QUANTIZATION: dynamic 8 bit int quantization. uses pytorch native utils- torch.quantization
- v) profile pruned and quantized model for INFERENCE (not TRAINING, but only the inference)



### Note

The demo requires a high-performance instance. We recommend running it on your personal workstation or provisioning a suitable cloud instance. On AWS, a g4dn.4xlarge instance is sufficient.

#### Recommended Specifications:

- vCPUs: 16
- GPU: 1 × NVIDIA T4 (16 GB VRAM)
- Memory (RAM): 64 GB
- CUDA / GPU Drivers: CUDA 11.8 or later

## 🚀 Import Required Libraries
This cell:
- Imports essential libraries for PyTorch operations, pruning, and quantization.
- Imports Hugging Face Transformers for model and tokenizer handling.
- Imports PyTorch Profiler for performance analysis.

In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.ao.quantization as tq
# import torch.quantization as tq

from transformers import AutoModelForCausalLM, AutoTokenizer, 
from transformers import BitsAndBytesConfig, QuantoConfig
from torch.profiler import profile, record_function, ProfilerActivity

# 🔧 Define Model and Profiling Settings
This cell:
- Specifies the model name to be used for pruning and profiling.
- Defines the pruning amount (`PRUNE_AMOUNT`) as 30%.
- Sets the batch size and maximum number of tokens to generate during inference.
- Provides a sample prompt for benchmarking.
- Specifies the directory to save profiler logs for the baseline model.

In [2]:
# MODEL_NAME      = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# This smaller model better on GEFORCE 3050
MODEL_NAME = "facebook/opt-350m"
PRUNE_AMOUNT    = 0.30      # 30% magnitude pruning
BATCH_SIZE      = 5
MAX_NEW_TOKENS  = 50
PROMPT          = (
    "In a world increasingly driven by artificial intelligence, the ability to interpret "
    "large language models efficiently is crucial for both research and deployment."
)
LOGDIR_BASELINE = "./logs/baseline"
LOGDIR_PRUNED_QUANTIZED = "./logs/pruned_quantized_cpu"

# 📦 Load Model and Tokenizer
This function:
- Loads the tokenizer and model using Hugging Face Transformers.
- Configures the model to use FP32 precision for accurate profiling.
- Moves the model to the specified device (CPU or GPU).
- Sets the model to evaluation mode to disable gradient computations.
- Returns the loaded tokenizer and model.

In [3]:
def load_model(model_name, device: torch.device):
    """
    TODO:
      - Load the tokenizer: AutoTokenizer.from_pretrained(model_name, use_fast=True)
      - Load the model: AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
      - Move model to `device` and call .eval()
      - Return (tokenizer, model)
    """
    # 1) Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    print("\n Sucessfully Loaded Tokenzier")
    
    # 2) Load FP32 model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32
    )
    print("\n Sucessfully Loaded Model")
    
    # 3) Move to device and set to eval mode
    model.to(device)
    model.eval()
    print("\n Sucessfully Moved Model to Device: ", device)

    # 4) Return both
    return tokenizer, model

# 📝 Create a Batch of Inputs
This function:
- Duplicates the provided prompt `BATCH_SIZE` times to simulate a batch of inputs.
- Tokenizes the batch with padding and truncation to ensure uniform input size.
- Moves the tokenized inputs to the specified device (CPU or GPU).
- Returns the prepared batch of inputs.

Note:
1) notice that the tokenizer can tokenize a single prompt or a batch of prompts in one go
2) tokenize a single prompt. Padding is unnecessary. Truncation can be added for safety
```
inputs = tokenizer(prompt,return_tensors="pt", truncation=True )
```
3) tokenize many prompts: padding is necessary for this. Truncation is always a good idea
```
# 1) Replicate prompt to create a Batch
texts = [prompt] * BATCH_SIZE

# 2) Tokenize with padding & truncation
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
```
4) Meaning of padding: lets say in batch there are many inputs,each of different length. all tokens are brought to the same length by padding the tokens that are too short (and possibly truncation if the token is too long)
```
Example
batch_input = ["Hi",
               "How are you",
               "It is a beautiful day to save lives".
               "Have fun",
               "Live long and prosper",
               "May the force be with you"
              ]
let say max_tokens = 16. Then all the above prompts are made 16 token each by padding.
```

5) Meaning of truncation: Some sentence (and hence its tokens) might be too long. In such a case the sentence /or it tokens are truncated
```
Example
max_tokens = 128
prompt ="This is the beginning of a very long document.......it spans research across several continents and across time.............this is the most detailed document report...."

This prompt has more than 128 tokens
so it will be truncated to 128 tokend
```

In [4]:
def make_batch(tokenizer, prompt: str, device: torch.device):
    """
    TODO:
      - Duplicate `prompt` BATCH_SIZE times into a list of strings
      - Tokenize with padding and truncation: tokenizer(..., return_tensors="pt")
      - Move inputs to `device`
      - Return the tokenized inputs
    """
    # 1) Replicate prompt
    texts = [prompt] * BATCH_SIZE

    # 2) Tokenize with padding & truncation
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # 3) Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    print("\n\n Sucessfully Created Batched Inputs")
    
    return inputs


def display_pandas_match_tensorboard(df, mode, top_k):
    #------------------------------------------------------------------------------------------
    ## CPU: Making profile.key_averages().table comparable to TensorBoard Operator View
    #------------------------------------------------------------------------------------------
    print("------- ", mode, " TOP OPERATIONS: CPU TIME (MATCHED WITH TENSORBOARD) --------")
    df_cpu_ops_only = df[
        ~df["Name"].str.startswith("cuda", na=False)
        & ~df["Name"].str.contains("ProfilerStep", na=False)
        & ~df["Name"].str.contains("enumerate", na=False)
    ]
    
    df_cpu_ops_only = df_cpu_ops_only.sort_values("CPU Time Total (ms)", ascending=False)
    
    display(pretty_display(df_cpu_ops_only.head(top_k)))
    
    
    #------------------------------------------------------------------------------------------
    ## GPU: Making profile.key_averages().table comparable to TensorBoard GPU Kernel View
    #------------------------------------------------------------------------------------------
    # rows that are actual GPU kernels:
    # - have CUDA self time
    # - are NOT aten ops
    # - are NOT cuda runtime API calls
    # - are NOT ProfilerStep
    print("------- ", mode, " TOP OPERATIONS: CUDA TIME (MATCHED WITH TENSORBOARD) --------")
    kernel_df = df[
        (df["CUDA Time Total (ms)"] > 0)
        & ~df["Name"].str.startswith("aten::", na=False)
        & ~df["Name"].str.startswith("cuda", na=False)
        & ~df["Name"].str.contains("ProfilerStep", na=False)
    ]
    
    kernel_df = kernel_df.sort_values("CUDA Time Total (ms)", ascending=False)
    
    display(pretty_display(kernel_df.head(top_k)))

## Helper functions to help print profiling results
- prof.key_averages.table() : displayed in pandas data frame
- prof.key_averages.table() matched with tensorboard : displayed in pandas data frame

In [5]:
def profiler_to_df(prof):
    events = prof.key_averages()

    total_self_cpu_us = sum(evt.self_cpu_time_total for evt in events)
    total_self_gpu_us = sum(
        getattr(evt, "self_device_time_total",
                getattr(evt, "self_cuda_time_total", 0))
        for evt in events
    )

    rows = []

    for evt in events:
        # ---------------- CPU TIME ----------------
        self_cpu_us = evt.self_cpu_time_total
        cpu_total_us = evt.cpu_time_total

        # ---------------- GPU TIME ----------------
        self_gpu_us = getattr(
            evt, "self_device_time_total",
            getattr(evt, "self_cuda_time_total", 0)
        )

        gpu_total_us = getattr(
            evt, "device_time_total",
            getattr(evt, "cuda_time_total", 0)
        )

        # ---------------- MEMORY (FIXED PATCH) ----------------
        cpu_mem = getattr(evt, "cpu_memory_usage", 0)
        cpu_self_mem = getattr(evt, "self_cpu_memory_usage", 0)

        gpu_mem = getattr(
            evt, "device_memory_usage",
            getattr(evt, "cuda_memory_usage", 0)
        )

        gpu_self_mem = getattr(
            evt, "self_device_memory_usage",
            getattr(evt, "self_cuda_memory_usage", 0)
        )

        rows.append({
            "Name": evt.key,

            # ---------------- CPU ----------------
            "CPU Time Self %": (self_cpu_us / total_self_cpu_us * 100) if total_self_cpu_us else 0,
            "CPU Time Self (ms)": self_cpu_us / 1000,

            "CPU Time Total %": (cpu_total_us / total_self_cpu_us * 100) if total_self_cpu_us else 0,
            "CPU Time Total (ms)": cpu_total_us / 1000,
            "CPU Time Avg (ms)": (cpu_total_us / evt.count / 1000) if evt.count else 0,

            # ---------------- GPU ----------------
            "CUDA Time Self %": (self_gpu_us / total_self_gpu_us * 100) if total_self_gpu_us else 0,
            "CUDA Time Self (ms)": self_gpu_us / 1000,

            "CUDA Time Total %": (gpu_total_us / total_self_gpu_us * 100) if total_self_gpu_us else 0,
            "CUDA Time Total (ms)": gpu_total_us / 1000,
            "CUDA Time Avg (ms)": (gpu_total_us / evt.count / 1000) if evt.count else 0,

            # ---------------- CALLS ----------------
            "Num Calls": evt.count,

            # ---------------- MEMORY ---------------            
            "CPU Mem Self (MB)": cpu_self_mem / (1024**2) if cpu_self_mem else 0,
            "CPU Mem Total (MB)": cpu_mem / (1024**2) if cpu_mem else 0,

            "CUDA Mem Total (MB)": gpu_mem / (1024**2) if gpu_mem else 0,
            "CUDA Mem Self (MB)": gpu_self_mem / (1024**2) if gpu_self_mem else 0,
        })

    return pd.DataFrame(rows)


# ---------------- STYLE FORMATTER ----------------
def pretty_display(df):
    return df.style.format({
        "CPU Time Self %": "{:.2f}",
        "CPU Time Self (ms)": "{:.3f}",
        "CPU Time Total %": "{:.2f}",
        "CPU Time Total (ms)": "{:.3f}",
        "CPU Time Avg (ms)": "{:.3f}",

        "CUDA Time Self %": "{:.2f}",
        "CUDA Time Self (ms)": "{:.3f}",
        "CUDA Time Total %": "{:.2f}",
        "CUDA Time Total (ms)": "{:.3f}",
        "CUDA Time Avg (ms)": "{:.3f}",

        "CPU Mem Self (MB)": "{:.2f}",
        "CPU Mem Total (MB)": "{:.2f}",        
        "CUDA Mem Self (MB)": "{:.2f}",
        "CUDA Mem Total (MB)": "{:.2f}",
    })

def display_pandas(prof, mode, top_k):
    df = profiler_to_df(prof)
    print("------- ", mode, " TOP OPERATIONS: by call --------")
    display(pretty_display(df.sort_values("Num Calls", ascending=False).head(top_k)))
    
    print("------- ", mode, " TOP OPERATIONS: CPU TIME  --------")
    display(pretty_display(df.sort_values("CPU Time Self (ms)", ascending=False).head(top_k)))
    
    print("------- ", mode, " TOP OPERATIONS: CPU MEMORY  --------")
    display(pretty_display(df.sort_values("CPU Mem Self (MB)", ascending=False).head(top_k)))
    
    if torch.cuda.is_available():
        print("------- ", mode, " TOP OPERATIONS: CUDA TIME  --------")
        display(pretty_display(df.sort_values("CUDA Time Self (ms)", ascending=False).head(top_k)))
    
        print("------- ", mode, " TOP OPERATIONS: CUDA MEMORY  --------")
        display(pretty_display(df.sort_values("CUDA Mem Self (MB)", ascending=False).head(top_k)))

    return df


def display_pandas_match_tensorboard(df, mode, top_k):
    #------------------------------------------------------------------------------------------
    ## CPU: Making profile.key_averages().table comparable to TensorBoard Operator View
    #------------------------------------------------------------------------------------------
    print("------- ", mode, " TOP OPERATIONS: CPU TIME (MATCHED WITH TENSORBOARD) --------")
    df_cpu_ops_only = df[
        ~df["Name"].str.startswith("cuda", na=False)
        & ~df["Name"].str.contains("ProfilerStep", na=False)
        & ~df["Name"].str.contains("enumerate", na=False)
    ]
    
    df_cpu_ops_only = df_cpu_ops_only.sort_values("CPU Time Total (ms)", ascending=False)
    
    display(pretty_display(df_cpu_ops_only.head(top_k)))
    
    
    #------------------------------------------------------------------------------------------
    ## GPU: Making profile.key_averages().table comparable to TensorBoard GPU Kernel View
    #------------------------------------------------------------------------------------------
    # rows that are actual GPU kernels:
    # - have CUDA self time
    # - are NOT aten ops
    # - are NOT cuda runtime API calls
    # - are NOT ProfilerStep
    print("------- ", mode, " TOP OPERATIONS: CUDA TIME (MATCHED WITH TENSORBOARD) --------")
    kernel_df = df[
        (df["CUDA Time Total (ms)"] > 0)
        & ~df["Name"].str.startswith("aten::", na=False)
        & ~df["Name"].str.startswith("cuda", na=False)
        & ~df["Name"].str.contains("ProfilerStep", na=False)
    ]
    
    kernel_df = kernel_df.sort_values("CUDA Time Total (ms)", ascending=False)
    
    display(pretty_display(kernel_df.head(top_k)))

# 📊 Profile Inference
This function:
- Profiles the model's inference performance using PyTorch Profiler.
- Captures:
  - CPU and CUDA activity.
  - Memory usage.
  - Operator-level performance metrics.
- Saves the profiling traces to the specified log directory for visualization in TensorBoard.
- Prints:
  - Top-3 operators by CPU self-time.
  - Top-3 operators by CUDA self-time.
  - Total CPU and CUDA self-times in milliseconds.

In [6]:
def profile_inference(model, inputs, logdir: str, label: str):
    """
    TODO:
      - Create `logdir` if it doesn't exist
      - Use `torch.profiler.profile` (CPU & CUDA, record_shapes, profile_memory, with_stack)
      - Inside the profiler, wrap the call to `model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)` in `record_function(label)`
      - After profiling, print:
          1) Top-3 ops by "self_cpu_time_total"
          2) Top-3 ops by "self_cuda_time_total"
          3) Total CPU vs CUDA self-time in milliseconds
      - Note: use `prof.key_averages().table(...)` and sum over `evt.self_cpu_time_total`, `evt.self_cuda_time_total`
      - Traces should be saved automatically by `tensorboard_trace_handler`
    """
    print("\n\n START: Profiling Inference . Will save to : " , logdir)
    
    # 1) Ensure log directory exists
    os.makedirs(logdir, exist_ok=True)

    # 2) Run profiler
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
        with_stack=True,
        on_trace_ready=torch.profiler.tensorboard_trace_handler(logdir)
    ) as prof:
        with record_function(label):
            _ = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

    # 3) Print profile inference tables
    top_k = 5
    mode = labl+" INFERENCE"
    df= display_pandas(prof, mode, top_k)
    display_pandas_match_tensorboard(df, mode, top_k)

    # 5) Summarize total CPU vs CUDA self times
    events     = prof.key_averages()
    total_cpu  = sum(evt.self_cpu_time_total for evt in events)
    total_cuda = sum(getattr(evt, "self_cuda_time_total", 0) for evt in events)
    print(f"\n=== {label} Total self-time ===")
    print(f"CPU  : {total_cpu/1e3:.2f} ms")
    print(f"CUDA : {total_cuda/1e3:.2f} ms")

    print(f"\nTrace files for '{label}' written to: {logdir}")

    print("\n\n END: Profiling Inference . Sucessfully saved logs to : " , logdir)

# ✂️ Apply Pruning and Quantization
This function:
- Applies unstructured magnitude-based pruning to all `nn.Linear` layers in the model.
  - Prunes 30% of the smallest-magnitude weights.
  - Makes the pruning masks permanent.
- Dynamically quantizes the pruned model to 8-bit integers (`torch.qint8`) for efficient inference.
- Returns the pruned and quantized model.

In [2]:
def prune_model(model_name):
    """
      - Pruning is done on the CPU , in place. Hence use a new copy of the model , loaded from the checkpoint
      - On CPU, apply `prune.l1_unstructured(..., amount=PRUNE_AMOUNT)` to every nn.Linear weight
      - Call `prune.remove(...)` to make masks permanent
      - Return the pruned model
    """

    # Pruning is done in place, in the CPU. 
    # Hence use a new copy of the model, loaded from the checkpoint
    tokenizer, pruned_model = load_model(model_name, torch.device("cpu"))
        
    # Prune on CPU    
    print("\n START: Pruning On CPU" )
    for module in pruned_model.modules():
        if isinstance(module, nn.Linear):
            # zero out PRUNE_AMOUNT fraction of the smallest‐magnitude weights
            prune.l1_unstructured(module, name="weight", amount=PRUNE_AMOUNT)
            # make the pruning permanent
            prune.remove(module, "weight")
    print("END: Pruning On CPU Successful!" )

    return pruned_model


def quantize_dynamic_cpu(model: nn.Module):
    """
      - Apply Pytorch Dynamic Quantization
      - It is performed on the CPU. It CANNOT be peformed on the GPU
      - It is Dynamic Quantization. Both weights and activations are quantized
      - Weights are quantized ahead of time . Activations are quantized during run time
      - Then apply `torch.quantization.quantize_dynamic` on {nn.Linear} with dtype=torch.qint8
      - During MATMUL = INT8 weights × INT8 activations

    Note:
    PyTorch describes quantize_dynamic as converting a float model to a dynamic, weight-only quantized model. 
    For qint8 Linear layers, activations are quantized dynamically during execution.
    """
    
    # Move model to cpu
    model.cpu()
    
    # Dynamic 8-bit quantization
    print("\n START: Quantization ON CPU" )
    quantized_model_dynamic = tq.quantize_dynamic(
        model,
        {nn.Linear},
        dtype=torch.qint8
    )
    print("\n END: Quantization on CPU Successful" )

    return quantized_model_dynamic


def quantize_dynamic_gpu(model_name: str):
    """
    Load a Hugging Face causal LM in 8-bit using bitsandbytes.
      - Apply Hugging Face : Bits and bytes Quantization
      - Notice in this the quantization is applied when loading the model itself using AutoModelForCausalLM 
      - This is CUDA-compatible quantization(i.e on the GPU)
      - It is a Hybrid Dynamic Quantization. 
          - Weights are quantized ahead of time .(to INT8) 
          - Most Activations are quantized during run time(to INT8)
          - selective higher-precision computation for sensitive values remain in FP32/FP16
    """

    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True
    )

    quantized_model_bnb = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )

    return quantized_model_bnb

In [ ]:
def quantize_weight_only_gpu(model_name: str):
    """
    Load a Hugging Face causal LM using weight-only quantization.

      - Weights are quantized, for example to INT8.
      - Activations are NOT quantized.
      - No activation calibration is used.
      - This is weight-only quantization, not dynamic quantization.
      - This is not full static quantization.
    """

    quanto_config = QuantoConfig(weights="int8")

    quantized_model_weight_only = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quanto_config,
        device_map="auto"
    )

    return quantized_model_weight_only


def quantize_full_static_cpu(model: nn.Module, calibration_loader):
    """
    Apply PyTorch post-training static quantization.

      - Runs on CPU.
      - Weights are quantized ahead of time.
      - Activations are quantized using calibration statistics.
      - Requires a representative calibration dataset.
      - This is full static quantization, not dynamic quantization.
      - Usually works better for CNN-style models than large Hugging Face LLMs.
    """

    model.cpu()
    model.eval()

    model.qconfig = tq.get_default_qconfig("fbgemm")

    prepared_model = tq.prepare(model, inplace=False)

    print("\nSTART: Static Quantization Calibration")
    with torch.no_grad():
        for batch in calibration_loader:
            prepared_model(**batch)
    print("END: Calibration Successful")

    quantized_model_static = tq.convert(prepared_model, inplace=False)

    return quantized_model_static

# 🚦 Start the Profiling Process
This function:
- Loads the model and tokenizer.
- Creates a batch of inputs using the provided prompt.
- Profiles the baseline model's inference performance.
- Saves the profiling results to the specified log directory.

# ▶️ Run the Profiling Pipeline
This cell:
- Calls the `start` function to execute the profiling pipeline.
- Outputs the profiling results, including operator-level performance metrics and total CPU/CUDA self-times.

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1a) Load & benchmark baseline
tokenizer, baseline_model = load_model(MODEL_NAME, device)

# 1b) Make inputs
batch_inputs_gpu = make_batch(tokenizer, PROMPT, device)
batch_inputs_cpu = {k: v.to(torch.device("cpu")) for k, v in batch_inputs.items()}

# 2) Prune Model  (in place , on cpu)
pruned_model = prune_model(MODEL_NAME)

# 3) Pytorch Dynamic Quantization (possible on CPU only) 
quantize_model_1 = quantize_dynamic_pytorch_cpu(baseline_model)

# 3) Bits&Bytes Hybrid Dynamic Quantization (GPU) 
quantize_model_2 = quantize_dynamic_bnb_gpu(MODEL_NAME)

In [9]:
print("PROFILE INFERENCE: Baseline Model")
profile_inference(baseline_model, batch_inputs_gpu, LOGDIR_BASELINE, label="Baseline")

print("PROFILE INFERENCE: Quantized Model")
profile_inference(pruned_quantized_model, batch_inputs, LOGDIR_PRUNED_QUANTIZED, label="Pruned_Quantized")


 Sucessfully Loaded Tokenzier


I0000 00:00:1777585537.321601   12065 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777585537.351528   12065 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777585538.075433   12065 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.



 Sucessfully Loaded Model


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



 Sucessfully Moved Model to Device:  cuda


 Sucessfully Created Batched Inputs
BEFORE: PRUNING & QUANTIZATION: PROFILE INFERENCE


 START: Profiling Inference . Will save to :  ./logs/baseline
-------  INFERENCE  TOP OPERATIONS: by call --------


...

-------  INFERENCE  TOP OPERATIONS: CPU TIME (MATCHED WITH TENSORBOARD) --------


,Name,CPU Time Self %,CPU Time Self (ms),CPU Time Total %,CPU Time Total (ms),CPU Time Avg (ms),CUDA Time Self %,CUDA Time Self (ms),CUDA Time Total %,CUDA Time Total (ms),CUDA Time Avg (ms),Num Calls,CPU Mem Self (MB),CPU Mem Total (MB),CUDA Mem Total (MB),CUDA Mem Self (MB)
0,Pruned_Quantized,34.68,560.444,100.00,1616.199,1616.199,0.00,0.000,0.00,0.000,0.000,1,-3101.81,0.00,0.00,0.00
39,quantized::linear_dynamic,43.23,698.656,44.75,723.256,0.098,0.00,0.000,0.00,0.000,0.000,7350,-389.48,390.52,0.00,0.00
50,aten::cat,8.48,137.115,9.45,152.738,0.062,0.00,0.000,0.00,0.000,0.000,2452,2343.01,2343.01,0.00,0.00
41,aten::scaled_dot_product_attention,0.24,3.822,4.69,75.757,0.063,0.00,0.000,0.00,0.000,0.000,1200,-0.53,35.18,0.00,0.00
42,aten::_scaled_dot_product_flash_attention_for_cpu,3.51,56.738,4.45,71.936,0.060,0.00,0.000,0.00,0.000,0.000,1200,-10.41,35.71,0.00,0.00


-------  INFERENCE  TOP OPERATIONS: CUDA TIME (MATCHED WITH TENSORBOARD) --------


,Name,CPU Time Self %,CPU Time Self (ms),CPU Time Total %,CPU Time Total (ms),CPU Time Avg (ms),CUDA Time Self %,CUDA Time Self (ms),CUDA Time Total %,CUDA Time Total (ms),CUDA Time Avg (ms),Num Calls,CPU Mem Self (MB),CPU Mem Total (MB),CUDA Mem Total (MB),CUDA Mem Self (MB)



=== Pruned_Quantized Total self-time ===
CPU  : 1616.21 ms
CUDA : 0.00 ms

Trace files for 'Pruned_Quantized' written to: ./logs/pruned_quantized_cpu


 END: Profiling Inference . Sucessfully saved logs to :  ./logs/pruned_quantized_cpu
